In [23]:
import unicodedata

import pandas as pd
from tensorflow.keras.utils import Sequence
from tensorflow.keras.layers import Conv2D,Dense,Dropout,Input,LSTM,Embedding
import os
import datasets
from datasets import Dataset,DatasetDict
import tensorflow as tf
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
with open ("D:/project_deeplearning/TEP.en-fa.en",encoding="utf-8") as f:
    en_s=f.read().splitlines()

with open ("D:/project_deeplearning/TEP.en-fa.fa",encoding="utf-8") as f:
    fa_s=f.read().splitlines()


assert len(en_s)==len(fa_s)

df=pd.DataFrame(
    {
        "en":en_s,
        "fa":fa_s
    }
)

data=pd.DataFrame({"train":Dataset.from_pandas(df)})
print(data["train"][0])

{'en': 'raspy breathing .', 'fa': 'صداي خر خر .'}


In [3]:
train=data["train"]
train[:10]

0    {'en': 'raspy breathing .', 'fa': 'صداي خر خر .'}
1                       {'en': 'dad .', 'fa': 'پدر .'}
2    {'en': 'maybe its the wind .', 'fa': 'شايد صدا...
3                         {'en': 'no .', 'fa': 'نه .'}
4    {'en': 'stop please stop .', 'fa': 'دست نگه دا...
5    {'en': 'you have a week , evans then well burn...
6                {'en': 'william .', 'fa': 'ويليام .'}
7    {'en': 'god damn it , william .', 'fa': 'لعنتي...
8    {'en': 'god damn it put that down .', 'fa': 'ل...
9               {'en': 'let go .', 'fa': 'بذار برم .'}
Name: train, dtype: object

In [4]:
def unicode_to_asci(s):
    return "".join(c for c in unicodedata.normalize("NFC",s) if unicodedata.category(c)!='MN' )

In [5]:
len(data)

612086

In [6]:
def preprossing(w):
    w=unicode_to_asci(w.lower().strip())
    w=re.sub(r"([.!?])",r"\1",w)
    w=re.sub(r'([""])+',"",w)
    w=w.rstrip().strip()
    w="<start>"+w+"<end>"
    return w

In [7]:
en_sen="Im very happy."
preprossing(en_sen)

'<start>im very happy.<end>'

In [8]:
fa_sen="درود بر تو."
preprossing(fa_sen)

'<start>درود بر تو.<end>'

In [14]:
df["en"]=df["en"].apply(preprossing)
df["fa"]=df["fa"].apply(preprossing)

In [46]:
vocab_size=2000
max_length=50
batch_size=64

token_en=Tokenizer(num_words=vocab_size,filters="")
token_fa=Tokenizer(num_words=vocab_size,filters="")



token_en.fit_on_texts(df["en"])
token_fa.fit_on_texts(df["fa"])

en_seq=token_en.texts_to_sequences(df["en"])
fa_seq=token_fa.texts_to_sequences(df["fa"])

en_seq=pad_sequences(en_seq , maxlen=max_length,padding="post")
fa_seq=pad_sequences(fa_seq,maxlen=max_length,padding="post")

decoder_inputs_array=fa_seq[:,:-1]
decoder_targets_array=fa_seq[:,1:]

dataset = tf.data.Dataset.from_tensor_slices(((en_seq, decoder_inputs_array), decoder_targets_array))
dataset = dataset.shuffle(buffer_size=len(en_seq)).batch(batch_size).prefetch(tf.data.AUTOTUNE)



In [47]:
(x,dec_in),y=next(iter(dataset))
print(x.shape,dec_in.shape,y.shape)

(64, 50) (64, 49) (64, 49)


In [48]:
latent_dim=256

encoder_inputs=Input(shape=(max_length,),name="encoder_inputs")
encoder_embedding=Embedding(input_dim=vocab_size,output_dim=latent_dim,mask_zero=True)(encoder_inputs)
encoder_output,state_h,state_c=LSTM(latent_dim,return_state=True)(encoder_embedding)



decoder_inputs=Input(shape=(None,),name="decoder_inputs")
decoder_embedding=Embedding(input_dim=vocab_size,output_dim=latent_dim,mask_zero=True,name="decoder_embedding")
decoder_embedd=decoder_embedding(decoder_inputs)
decoder_lstm=LSTM(latent_dim,return_sequences=True,return_state=True,name="decoder_lstm")
decoder_outputs,state_h_dec,state_c_dec=decoder_lstm(decoder_embedd,initial_state=[state_h,state_c])




In [49]:
decoder_dense=Dense(vocab_size,activation="softmax")
decoder_outputs=decoder_dense(decoder_outputs)

In [50]:
model=tf.keras.Model([encoder_inputs,decoder_inputs],decoder_outputs)
model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [ ]:
history=model.fit([en_seq,decoder_inputs_array],decoder_targets_array,
                  batch_size=batch_size,epochs=20)

Epoch 1/20
9564/9564 ━━━━━━━━━━━━━━━━━━━━ 4616s 483ms/step - accuracy: 0.3696 - loss: 3.6357
Epoch 2/20
9564/9564 ━━━━━━━━━━━━━━━━━━━━ 1959s 202ms/step - accuracy: 0.4291 - loss: 2.9642
Epoch 3/20
9564/9564 ━━━━━━━━━━━━━━━━━━━━ 1428s 149ms/step - accuracy: 0.4513 - loss: 2.7683
Epoch 4/20
4494/9564 ━━━━━━━━━━━━━━━━━━━━ 12:25 147ms/step - accuracy: 0.4655 - loss: 2.6504